# Walk-forward baselines

The no-model guesses from the EDA, scored on six test years instead of one. For each test year Y: train on every year up to Y−2, tune on Y−1, test on Y. The guesses don't need training or tuning; models will.

In [1]:
import numpy as np
import pandas as pd

from data import load_all
from evaluation import TEST_YEARS, baseline_guesses, score, walk_forward_splits

markets = load_all()  # cleaned and checked
order = list(markets)

## The splits

The same calendar years for every market. Day counts for Indonesia:

In [2]:
df = markets["Indonesia"]
pd.DataFrame([
    {
        "test year": year,
        "train years": f"{df.index[train].year.min()}–{df.index[train].year.max()}",
        "tune year": year - 1,
        "train days": train.sum(),
        "tune days": tune.sum(),
        "test days": test.sum(),
    }
    for year, train, tune, test in walk_forward_splits(df.index)
]).set_index("test year")

,train years,tune year,train days,tune days,test days
test year,,,,,
2017,2012–2015,2016,976,246,238
2018,2012–2016,2017,1222,238,240
2019,2012–2017,2018,1460,240,245
2020,2012–2018,2019,1700,245,242
2021,2012–2019,2020,1945,242,247
2022,2012–2020,2021,2187,247,246


## Baselines on each test year

Only days with a return are scored (the day after Singapore's missing months has none).

In [3]:
rows = []
for market, df in markets.items():
    actual = df["ret"] > 0
    has_return = df["ret"].notna().to_numpy()
    guesses = baseline_guesses(df)
    for year, train, tune, test in walk_forward_splits(df.index):
        test = test & has_return
        for guess in guesses:
            rows.append({"market": market, "year": year, "guess": guess,
                         **score(actual[test], guesses[guess][test])})

scores = pd.DataFrame(rows)
scores.head()

,market,year,guess,days,accuracy %,balanced acc %,MCC
0,Indonesia,2017,always up,238,55.882353,50.000000,0.000000
1,Indonesia,2017,always down,238,44.117647,50.000000,0.000000
2,Indonesia,2017,same as yesterday,238,53.361345,52.756892,0.055085
3,Indonesia,2018,always up,240,51.250000,50.000000,0.000000
4,Indonesia,2018,always down,240,48.750000,50.000000,0.000000


### Bar to beat: the best guess's accuracy %, by test year

In [4]:
bar = scores.pivot_table(index="market", columns="year", values="accuracy %", aggfunc="max").reindex(order)
bar["range"] = bar.max(axis=1) - bar.min(axis=1)
bar.round(1)

year,2017,2018,2019,2020,2021,2022,range
market,,,,,,,
Indonesia,55.9,51.2,53.9,52.5,53.0,52.0,4.6
Malaysia,51.9,55.1,53.7,51.2,50.6,53.9,4.5
Philippines,55.6,53.5,52.1,50.4,52.4,52.2,5.1
Singapore,51.2,52.2,50.8,54.5,54.5,54.8,4.0
Thailand,56.0,51.8,51.6,51.9,53.5,56.0,4.4
India,56.3,54.1,53.9,59.1,56.9,59.3,5.4


### Which guess sets the bar

In [5]:
best = scores.loc[scores.groupby(["market", "year"])["accuracy %"].idxmax()]
best.pivot(index="market", columns="year", values="guess").reindex(order)

year,2017,2018,2019,2020,2021,2022
market,,,,,,
Indonesia,always up,always up,same as yesterday,always up,always up,always up
Malaysia,always up,same as yesterday,always down,always up,always down,same as yesterday
Philippines,always up,always down,always up,always up,always up,always up
Singapore,always up,always down,always up,always down,always up,always up
Thailand,always up,same as yesterday,same as yesterday,always up,always up,same as yesterday
India,always up,always up,same as yesterday,always up,same as yesterday,same as yesterday


### Same as yesterday: balanced accuracy % and MCC

The constant guesses always score 50% and 0, so only this guess is shown.

In [6]:
same = scores[scores["guess"] == "same as yesterday"]
same.pivot(index="market", columns="year", values="balanced acc %").reindex(order).round(1)

year,2017,2018,2019,2020,2021,2022
market,,,,,,
Indonesia,52.8,50.0,53.7,48.6,46.4,44.6
Malaysia,48.1,55.0,41.5,46.7,49.8,53.8
Philippines,45.5,53.3,46.2,48.3,49.0,50.5
Singapore,50.8,51.3,48.0,46.0,44.2,52.0
Thailand,49.9,51.8,51.6,47.6,47.9,55.9
India,51.5,52.5,53.7,53.2,56.3,59.3


In [7]:
same.pivot(index="market", columns="year", values="MCC").reindex(order).round(2)

year,2017,2018,2019,2020,2021,2022
market,,,,,,
Indonesia,0.06,-0.00,0.07,-0.03,-0.07,-0.11
Malaysia,-0.04,0.10,-0.17,-0.07,-0.00,0.08
Philippines,-0.09,0.07,-0.08,-0.03,-0.02,0.01
Singapore,0.02,0.03,-0.04,-0.08,-0.12,0.04
Thailand,-0.00,0.04,0.03,-0.05,-0.04,0.12
India,0.03,0.05,0.07,0.06,0.13,0.19


## How big does a gap need to be?

Two kinds of noise:
- **Year to year:** how much the bar itself moves between test years.
- **Within the test days:** the standard error of an accuracy near 50%, which is 50 / √(number of days). Pooling all six test years makes it much smaller than for one year.

A model that beats the bar by less than about two standard errors could be doing so by luck.

In [8]:
years = list(TEST_YEARS)
days_per_year = scores[scores["guess"] == "always up"].pivot(index="market", columns="year", values="days").reindex(order)

pd.DataFrame({
    "bar: mean %": bar[years].mean(axis=1),
    "bar: std across years": bar[years].std(axis=1),
    "bar: range": bar["range"],
    "1 s.e., one year": 50 / np.sqrt(days_per_year.median(axis=1)),
    "1 s.e., six years pooled": 50 / np.sqrt(days_per_year.sum(axis=1)),
}).round(1)

,bar: mean %,bar: std across years,bar: range,"1 s.e., one year","1 s.e., six years pooled"
market,,,,,
Indonesia,53.1,1.6,4.6,3.2,1.3
Malaysia,52.7,1.8,4.5,3.2,1.3
Philippines,52.7,1.7,5.1,3.2,1.3
Singapore,53.0,1.8,4.0,3.2,1.4
Thailand,53.5,2.1,4.4,3.2,1.3
India,56.6,2.3,5.4,3.2,1.3
